# 01 — Estatística descritiva: versão mínima

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/01_estatistica_descritiva_vers%C3%A3o_minima.ipynb)

**Duração estimada:** 15–20 minutos. Execute as quatro células de código em ordem no Colab ou em Jupyter com `pandas` e `matplotlib` instalados. O carregamento requer internet.

## Fonte e licença

[CDC Diabetes Health Indicators — UCI 891](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators). Consulte a fonte para os códigos das variáveis e os termos de uso.

Usaremos `BMI` (índice de massa corporal, IMC), uma variável numérica, e `Diabetes_binary`, uma variável categórica: **0 = sem diabetes; 1 = pré-diabetes/diabetes**.

Material com finalidade exclusivamente educacional. Os indicadores incluem autorrelato; os resultados descrevem esta amostra e não devem orientar decisões clínicas.

## 1. Carregar e inspecionar os dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

dados = pd.read_csv(
    "https://archive.ics.uci.edu/static/public/891/data.csv", index_col="ID"
)
print(f"Linhas: {len(dados):,} | Colunas: {dados.shape[1]}")
display(dados[["BMI", "Diabetes_binary"]].head())
display(dados[["BMI", "Diabetes_binary"]].isna().sum().to_frame("Ausências"))

## 2. Resumir uma variável numérica

`describe()` mostra contagem (`count`), média (`mean`), desvio-padrão (`std`), mínimo, quartis e máximo. **50% é a mediana**; **IQR = 75% − 25%** mede a amplitude dos 50% centrais. Compare média e mediana e observe a distribuição no histograma.

In [ ]:
resumo = dados["BMI"].describe().to_frame("IMC")
resumo.loc["IQR"] = resumo.loc["75%"] - resumo.loc["25%"]
display(resumo.round(2))

ax = dados["BMI"].plot.hist(bins=30, figsize=(7, 4), edgecolor="white")
ax.set(title="Distribuição do IMC", xlabel="IMC (kg/m²)", ylabel="Número de registros")
plt.tight_layout()
plt.show()

## 3. Resumir uma variável categórica

Para categorias, use contagens e percentuais. Os códigos 0 e 1 identificam grupos. As barras mostram a frequência do indicador **nesta amostra**, sem estimar a prevalência na população.

In [ ]:
rotulos = {0: "Sem diabetes", 1: "Pré-diabetes/diabetes"}
frequencias = (
    dados["Diabetes_binary"].value_counts().sort_index()
    .rename(index=rotulos).to_frame("n")
)
frequencias["Percentual (%)"] = 100 * frequencias["n"] / frequencias["n"].sum()
display(frequencias.round(1))

ax = frequencias["Percentual (%)"].plot.bar(figsize=(7, 4), rot=0)
ax.set(title="Indicador de diabetes na amostra", xlabel="", ylabel="Percentual (%)", ylim=(0, 100))
plt.tight_layout()
plt.show()

## 4. Comparar o IMC entre grupos

O boxplot mostra mediana (linha), quartis (caixa) e valores além dos limites dos bigodes (pontos); esses pontos não são necessariamente erros. Compare a posição e a dispersão das distribuições. **Diferenças entre grupos não demonstram causalidade.**

In [ ]:
por_grupo = dados.groupby("Diabetes_binary")["BMI"].agg(
    n="count", media="mean", mediana="median", desvio_padrao="std"
)
display(por_grupo.rename(index=rotulos).round(2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([
    dados.loc[dados["Diabetes_binary"] == codigo, "BMI"].dropna()
    for codigo in rotulos
])
ax.set_xticks([1, 2], labels=list(rotulos.values()))
ax.set(title="IMC por indicador de diabetes", ylabel="IMC (kg/m²)")
plt.tight_layout()
plt.show()

**Para praticar:** qual grupo tem maior mediana de IMC? As distribuições se sobrepõem? Escreva uma conclusão descritiva e uma limitação da análise.